# E00 — Dataset and Split Validation

Reconstruction-v2 analysis notebook. Local-only: reads the official ContractNLI split files and
this experiment's own saved `results/*.json` — no LLM/API/embedding-model calls. Reusable logic
(metric implementations, matching, label mapping) is imported from `evaluation/`, not
reimplemented here (per `docs/experiment_protocol.md`'s notebooks-vs-source-code rule).

See `README.md` (experiment template) and `summary.md` (full write-up + decision) in this same
directory, and `docs/evaluation_protocol.md` Part 1 for the frozen protocol this experiment
established.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E00_dataset_validation" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from evaluation.metrics import (
    _case_key,
    _match_predictions_to_golds,
    joint_label_evidence_correctness,
    recall_with_ci,
)
from evaluation.schemas import GoldCase, Label, Prediction

print("repo root:", REPO_ROOT)

repo root: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Official split counts and label distribution

In [2]:
splits = {}
for split in ["train", "dev", "test"]:
    with open(REPO_ROOT / f"data/contractnli/{split}.json") as f:
        d = json.load(f)
    docs = d["documents"]
    cls_counts = Counter()
    evidence_by_class = Counter()
    total_cases = 0
    for doc in docs:
        for hyp, ann in doc["annotation_sets"][0]["annotations"].items():
            total_cases += 1
            cls_counts[ann["choice"]] += 1
            if ann["spans"]:
                evidence_by_class[ann["choice"]] += 1
    splits[split] = {
        "documents": len(docs), "total_cases": total_cases,
        "class_counts": dict(cls_counts), "evidence_by_class": dict(evidence_by_class),
    }
    print(f"{split:5s}  docs={len(docs):4d}  cases={total_cases:5d}  "
          f"classes={dict(cls_counts)}")

train  docs= 423  cases= 7191  classes={'NotMentioned': 2820, 'Entailment': 3530, 'Contradiction': 841}
dev    docs=  61  cases= 1037  classes={'Entailment': 519, 'Contradiction': 95, 'NotMentioned': 423}
test   docs= 123  cases= 2091  classes={'NotMentioned': 903, 'Entailment': 968, 'Contradiction': 220}


## 2. NotMentioned evidence policy — verify empty-spans-by-definition directly

In [3]:
for split, s in splits.items():
    nm_with_evidence = s["evidence_by_class"].get("NotMentioned", 0)
    nm_total = s["class_counts"].get("NotMentioned", 0)
    ec_total = s["class_counts"].get("Entailment", 0) + s["class_counts"].get("Contradiction", 0)
    ec_with_evidence = (s["evidence_by_class"].get("Entailment", 0)
                         + s["evidence_by_class"].get("Contradiction", 0))
    print(f"{split:5s}  NotMentioned-with-evidence: {nm_with_evidence}/{nm_total}   "
          f"Entailment+Contradiction-with-evidence: {ec_with_evidence}/{ec_total}")
    assert nm_with_evidence == 0, "NotMentioned must never carry gold evidence"
    assert ec_with_evidence == ec_total, "Entailment/Contradiction must always carry gold evidence"
print("\nConfirmed across all 3 splits: NotMentioned evidence-absence is definitional, not invented.")

train  NotMentioned-with-evidence: 0/2820   Entailment+Contradiction-with-evidence: 4371/4371
dev    NotMentioned-with-evidence: 0/423   Entailment+Contradiction-with-evidence: 614/614
test   NotMentioned-with-evidence: 0/903   Entailment+Contradiction-with-evidence: 1188/1188

Confirmed across all 3 splits: NotMentioned evidence-absence is definitional, not invented.


## 3. Split overlap check (document-level leakage)

In [4]:
ids_by_split, names_by_split = {}, {}
for split in ["train", "dev", "test"]:
    with open(REPO_ROOT / f"data/contractnli/{split}.json") as f:
        d = json.load(f)
    ids_by_split[split] = {doc["id"] for doc in d["documents"]}
    names_by_split[split] = {doc["file_name"] for doc in d["documents"]}

pairs = [("train", "dev"), ("train", "test"), ("dev", "test")]
for a, b in pairs:
    id_overlap = len(ids_by_split[a] & ids_by_split[b])
    name_overlap = len(names_by_split[a] & names_by_split[b])
    print(f"{a} ∩ {b}: doc-id overlap = {id_overlap}, filename overlap = {name_overlap}")
    assert id_overlap == 0 and name_overlap == 0

train ∩ dev: doc-id overlap = 0, filename overlap = 0
train ∩ test: doc-id overlap = 0, filename overlap = 0
dev ∩ test: doc-id overlap = 0, filename overlap = 0


## 4. Evidence-threshold audit — gold span-count distribution

Empirical basis for keeping `tau_evidence=0.5` as the (not-yet-scientifically-frozen) interim
default — see `docs/evaluation_protocol.md` Part 1 section 16.

In [5]:
span_len_counts = Counter()
with open(REPO_ROOT / "data/contractnli/test.json") as f:
    d = json.load(f)
for doc in d["documents"]:
    for hyp, ann in doc["annotation_sets"][0]["annotations"].items():
        if ann["choice"] in ("Entailment", "Contradiction"):
            span_len_counts[len(ann["spans"])] += 1

total = sum(span_len_counts.values())
single_span_share = span_len_counts[1] / total
mean_spans = sum(k * v for k, v in span_len_counts.items()) / total
print("gold span-count distribution (test split):", dict(sorted(span_len_counts.items())))
print(f"share of cases with exactly 1 gold span: {single_span_share:.1%}")
print(f"mean gold spans per case: {mean_spans:.2f}")
print("\ntau_evidence=0.5 => full coverage required when n=1 (the plurality of cases),")
print("majority-of-gold-spans rule when n>=2. Interim default, not sensitivity-tested (E06's job).")

gold span-count distribution (test split): {1: 519, 2: 404, 3: 133, 4: 61, 5: 33, 6: 19, 7: 15, 8: 1, 10: 1, 14: 1, 18: 1}
share of cases with exactly 1 gold span: 43.7%
mean gold spans per case: 2.02

tau_evidence=0.5 => full coverage required when n=1 (the plurality of cases),
majority-of-gold-spans rule when n>=2. Interim default, not sensitivity-tested (E06's job).


## 5. Metric sanity checks — hand-built cases (no dataset, no model)

In [6]:
def gold(doc_id, hyp_id, label, spans=None, split=""):
    return GoldCase(doc_id=doc_id, hypothesis_id=hyp_id, gold_label=Label(label),
                     gold_span_indices=spans or [], split=split)

def pred(doc_id, hyp_id, label, spans=None, split=""):
    return Prediction(doc_id=doc_id, hypothesis_id=hyp_id, predicted_label=Label(label),
                       retrieved_span_indices=spans or [], split=split)

cases = [
    ("correct label + correct evidence", [gold("d", "h1", "Entailment", [0, 1])],
     [pred("d", "h1", "Entailment", [0])], 1.0),
    ("correct label + wrong evidence", [gold("d", "h1", "Entailment", [0, 1, 2, 3])],
     [pred("d", "h1", "Entailment", [0])], 0.0),
    ("wrong label + correct evidence", [gold("d", "h1", "Contradiction", [0, 1])],
     [pred("d", "h1", "Entailment", [0, 1])], 0.0),
    ("NotMentioned + empty evidence", [gold("d", "h1", "NotMentioned")],
     [pred("d", "h1", "NotMentioned", [])], 1.0),
    ("NotMentioned + fabricated evidence", [gold("d", "h1", "NotMentioned")],
     [pred("d", "h1", "NotMentioned", [3])], 0.0),
]
for name, golds, preds, expected in cases:
    actual = joint_label_evidence_correctness(preds, golds)
    status = "OK" if actual == expected else "MISMATCH"
    print(f"[{status}] {name}: expected {expected}, got {actual}")

[OK] correct label + correct evidence: expected 1.0, got 1.0
[OK] correct label + wrong evidence: expected 0.0, got 0.0
[OK] wrong label + correct evidence: expected 0.0, got 0.0
[OK] NotMentioned + empty evidence: expected 1.0, got 1.0
[OK] NotMentioned + fabricated evidence: expected 0.0, got 0.0


## 6. Case-ID split-qualification — legacy compatibility demo

In [7]:
# Legacy (no split set) still matches exactly as before.
legacy_golds = [gold("d1", "h1", "Entailment", [0])]
legacy_preds = [pred("d1", "h1", "Entailment", [0])]
print("legacy match count:", len(_match_predictions_to_golds(legacy_preds, legacy_golds)))

# Split-qualified records only match the identical split -- a doc_id collision across
# two different splits no longer silently merges.
train_gold = [gold("d1", "h1", "Entailment", [0], split="train")]
dev_pred = [pred("d1", "h1", "Contradiction", [0], split="dev")]
print("cross-split (same doc_id) match count:", len(_match_predictions_to_golds(dev_pred, train_gold)))

legacy match count: 1
cross-split (same doc_id) match count: 0


## 7. Contamination summary (loaded, not recomputed)

In [8]:
with open(REPO_ROOT / "experiments/E00_dataset_validation/results/contamination_summary.json") as f:
    contamination = json.load(f)
print(json.dumps(contamination["role_conflation_finding"], indent=2))
print()
print("Historical experiments that touched official TEST:")
for item in contamination["historical_experiments_that_touched_official_TEST"]:
    print(" -", item)

"Historically DEV was used for both development/tuning and validation purposes simultaneously via the reused 150-case sample -- disclosed in docs/data_contamination_register.md section 3 as a methodological limitation of the historical T-series work. Reconstruction-v2 uses the official TRAIN/DEV/TEST split as-is (TRAIN=development/tuning, DEV=validation/architecture selection, TEST=final evaluation) and does its own independent DEV-based selection going forward -- this disclosed history does not reassign DEV's role (see docs/data_contamination_register.md section 4 for why a fourth split was considered and rejected)."

Historical experiments that touched official TEST:
 - scripts/run_final_test_evaluation.py (T041-A 500-case subsample, T041-B full 2091 cases)
 - scripts/run_full_rule_baseline_test.py (full 2091 cases, zero-cost deterministic rule baseline)
 - scripts/run_hosted_comparison.py (hosted-vs-local, same seed=42 subsample as T041)
 - scripts/analyze_test_set_results.py (read-

## 8. Conclusion / decision

**Result: PASS.** Official splits are clean (zero doc-level overlap), NotMentioned's
evidence-absence is definitional (verified directly, not assumed), and the metric implementations
behave correctly on every hand-constructed sanity case, including the corrected NotMentioned
joint-metric policy and the new split-qualified case-ID matching.

**Decision:** proceed to E00B (budget/runtime forecast) once approved. Full write-up:
`summary.md` in this directory. Full protocol: `docs/evaluation_protocol.md` Part 1.